# [재실행 전용 · 2026-10-04 회계 수정] crypto_beta_robustness_colab.ipynb

원본 `crypto_beta_robustness_colab.ipynb`에서 **이번 재실행에 필요한 셀만** 뽑은 노트북이다. 학습 셀(`.learn`)은 빠져 있고, 저장된 모델을 불러와 평가만 한다.
위에서부터 **순서대로 전부 실행**하면 된다. 결과 CSV는 원본과 같은 이름으로 `data/crypto/`에 저장된다(실행 전 백업 셀 먼저).

결과: 6.1 다중 시드·6.2 DSR·6.4 앙상블.


In [ ]:
import shutil, glob, os
from google.colab import drive
drive.mount('/content/drive')
src = '/content/drive/MyDrive/졸업프로젝트/data/crypto'
dst = src + '/_pre_accounting_fix_20261004'
if not os.path.exists(dst):
    os.makedirs(dst)
    for p in glob.glob(src + '/*.csv') + glob.glob(src + '/*.png'):
        shutil.copy2(p, dst)
print('백업 폴더:', dst, len(os.listdir(dst)), '개')


# β=-30 배포 구성 강건성 검증 — 다중 시드 (`paper/GAP_ANALYSIS.md` §1.1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


## 1. 설정 — 데이터 경로, Stage A 하이퍼파라미터, 이 노트북에서 검증할 배포 구성

In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'

RL_EMB_H5 = CRYPTO_DIR / 'crypto_rl_embeddings.h5'

EMB_DIM    = 64
TC         = 0.001    # 거래비용 (Binance taker fee 기준) — crypto_hrl_earnhft_colab.ipynb와 동일
BUCKET_SEC = 1800     # 30분 버킷

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'
VALID_START, VALID_END = '2025-10-01', '2025-12-31'   # §14-A와 동일 분할(학습 노출 구간 — 완전한 홀드아웃 아님)

# --- Stage A 하이퍼파라미터 (crypto_hrl_earnhft_colab.ipynb와 동일) ---
N_LABELS       = 5
RISK_THRESHOLD = 0.2
LOWPASS_WINDOW = 48

# --- 이 노트북에서 검증할 배포 구성 (crypto/inference/policy_v4.py와 동일) ---
BETA                = -30
TOP_K               = 10
ALPHA               = 0.01
POOL_TIMESTEPS_EACH = 100_000

# --- 다중 시드 (기존 배포 에이전트는 ppo seed=42 / chunk rng seed=2001로 학습됨 — 아래 5개는 신규) ---
SEEDS = [11, 22, 33, 44, 55]

POOL_DIR_ROBUST      = MODEL_DIR / 'crypto_beta_robustness'
DEPLOYED_AGENT_PATH  = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'  # 현재 라이브 배포 중인 seed=42 에이전트

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음 — crypto_hybrid_model_colab.ipynb 먼저 실행'
assert DEPLOYED_AGENT_PATH.exists(), f'배포 중인 에이전트 없음: {DEPLOYED_AGENT_PATH}'
print('경로 확인 완료')


## 2. 데이터 로드 (crypto_hrl_earnhft_colab.ipynb §1~2와 동일)

In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]     # epoch seconds (UTC)
    embs_all    = f['embeddings'][:]     # (N, 64)

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

emb_index  = {}
sym_ts_set = {}
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9
    y     = df['RetTarget_6b'].values
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


## 3. CryptoPortfolioEnv (crypto_hrl_earnhft_colab.ipynb §3과 동일, 변경 없음)

In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    """
    크립토 44종목 개별 자산 기반 포트폴리오 운용 환경 (클러스터링 없음)

    State:  종목별 임베딩(N×64) + 현재 비중(N) → N*64+N dim
    Action: 종목별 목표 포트폴리오 비중 (N-dim, Softmax 정규화)
    Reward: 30분 포트폴리오 수익률(RetTarget_6b, 이미 forward-looking) − 거래비용
    스텝 = 30분 버킷(BUCKET_SEC) 단위 — emb_index/ret_index 모두 (symbol, bucket) 키.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)

    def _asset_ret(self, bucket):
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _tradable(self, bucket):
        """해당 버킷에 수익률 데이터가 있는(거래 가능한) 종목 마스크"""
        return np.array([(sym, bucket) in self.ret_index for sym in self.symbols])

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = _mask_untradable(np.ones(self.n_assets, dtype=np.float32),
                                          self._tradable(self.buckets[0]))  # 거래 가능 종목 등가중에서 시작
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        action  = np.exp(action - action.max())
        action  = action / action.sum()
        action  = _mask_untradable(action, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        turnover = float(np.abs(action - self.weights).sum())
        tc_cost  = self.tc * turnover

        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


## 4. 버킷 분할 — train / valid(2025 Q4) / test

In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
valid_buckets = [b for b in train_buckets if VALID_START <= bucket_to_date(b) <= VALID_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]

print(f'학습 기간: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'valid 기간: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')
print(f'test 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(train_buckets[:100], **env_kwargs), warn=True)
print('OK')


## 5. Stage A — 시장 레짐 분할 (crypto_hrl_earnhft_colab.ipynb §4와 동일)

In [ ]:
# crypto/market_segmentation.py 이식 (Colab은 저장소를 마운트하지 않으므로 동일 코드를 셀에 포함)
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    """가격 시계열을 레짐 구간으로 분할하고 라벨을 부여한다 (EarnHFT Algorithm 3 이식)."""
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다 (최소 3개 포인트 필요)')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    """EarnHFT Eq.4 이식 — chunk 기울기(r)에 대한 β-지수가중 샘플링 우선순위(합 1)."""
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


In [ ]:
eq_ret_per_bucket = np.array([
    np.mean([ret_index.get((sym, b), 0.0) for sym in symbols])
    for b in train_buckets
])
eq_nav = pd.Series(np.cumprod(1 + eq_ret_per_bucket), index=range(len(train_buckets)))

seg = segment_and_label(
    eq_nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD,
    lowpass_window=LOWPASS_WINDOW,
)
print(f"구간 수: {len(seg.chunks)} (평균 길이 {seg.chunks['length'].mean():.0f}버킷 ≈ {seg.chunks['length'].mean()/48:.1f}일)")
print(seg.chunks['regime'].value_counts())


## 6. 저수준 환경 — SparseGatedCryptoPortfolioEnv + ChunkedSparseGatedEnv (§13-A/B와 동일)

In [ ]:
class SparseGatedCryptoPortfolioEnv(CryptoPortfolioEnv):
    """
    FreQuant(top-K 희소 선택) + DeepClair(전용 리스크게이트 ρ) 결합 저수준 환경.
    crypto_hrl_earnhft_colab.ipynb §13-A와 동일 — 배포 구성(policy_v4.py)이 실제로 쓰는 행동 공간.
    """

    def __init__(self, *args, top_k=TOP_K, **kwargs):
        self.top_k = top_k
        super().__init__(*args, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def action_to_weights(self, action):
        action = np.asarray(action, dtype=np.float64)
        asset_logits, gate_logit = action[:-1], action[-1]
        asset_logits = np.where(self._tradable(self.buckets[self.t]), asset_logits, -np.inf)  # 거래 불가 종목 제외
        k = min(self.top_k, self.n_assets)
        top_idx = np.argpartition(asset_logits, -k)[-k:]
        sel = asset_logits[top_idx]
        sel = np.exp(sel - sel.max())
        sel = sel / sel.sum()
        risky_w = np.zeros(self.n_assets, dtype=np.float32)
        risky_w[top_idx] = sel
        rho = float(1.0 / (1.0 + np.exp(-gate_logit)))
        return (rho * risky_w).astype(np.float32), rho, top_idx

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.rho_history = []
        self.selection_history = []
        return obs, info

    def step_with_weights(self, target_w, extra_info=None):
        target_w = _mask_untradable(target_w, self._tradable(self.buckets[self.t]), renorm=False)  # 빠진 몫은 현금
        turnover = float(np.abs(target_w - self.weights).sum())
        tc_cost  = self.tc * turnover
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((target_w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(target_w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': turnover, 'tc_cost': tc_cost}
        if extra_info:
            info.update(extra_info)
            if 'rho' in extra_info:
                self.rho_history.append(extra_info['rho'])
            if 'selected' in extra_info:
                self.selection_history.append(extra_info['selected'])
        return self._get_obs(), reward, done, False, info

    def step(self, action):
        target_w, rho, top_idx = self.action_to_weights(action)
        return self.step_with_weights(target_w, extra_info={'rho': rho, 'selected': top_idx.tolist()})


class ChunkedSparseGatedEnv(SparseGatedCryptoPortfolioEnv):
    """§5의 β-우선순위 청크 샘플링, 저수준 환경만 v4로 교체 — Stage B 학습에 사용."""

    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, top_k=TOP_K, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], top_k=top_k, **env_kwargs)

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


# 스모크 테스트
_sg_env = SparseGatedCryptoPortfolioEnv(test_buckets[:200], top_k=TOP_K, **env_kwargs)
check_env(_sg_env, warn=True)
obs, _ = _sg_env.reset()
a = np.random.randn(_sg_env.n_assets + 1).astype(np.float32)
obs, r, done, _, info = _sg_env.step(a)
print(f"action space: {_sg_env.action_space.shape} | 선택된 종목 수: {len(info['selected'])} | "
      f"rho: {info['rho']:.3f} | reward: {r:.5f}")


## 7. 성과 계산 함수

In [ ]:
def compute_perf(values, steps_per_year=48 * 365):
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {'총 수익률': total_ret, '연환산 수익률': annual_ret, 'Sharpe': sharpe, 'MDD': max_dd}


def equal_weight_backtest(env):
    obs, _ = env.reset()
    done, values = False, [1.0]
    action = np.ones(env.n_assets, dtype=np.float32) / env.n_assets
    while not done:
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


def run_beta_smoothed(model, alpha, buckets, top_k=TOP_K):
    """β 에이전트 하나를 라우터 없이 buckets 위에서 실행 + 관성 스무딩
    (crypto/inference/policy_v4.py의 라이브 로직과 동일한 매 스텝 블렌딩)."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    return np.array(env.value_history), list(env.turnover_history)


def perf_row(values, turnovers, tc=TC):
    perf = compute_perf(values)
    t = np.asarray(turnovers, dtype=np.float64)
    perf['평균 턴오버/스텝'] = t.mean()
    perf['누적 비용 드래그'] = 1 - np.prod(1 - tc * t)
    return perf


## 8. 다중 시드 학습 — β=-30 에이전트 5개 재학습 (Stage B, §13-C와 동일 예산·하이퍼파라미터)

In [ ]:
# 재실행용 로더 — 세션이 끊겨도 저장된 시드 모델만 불러와 이어서 평가 가능 (idempotent)
POOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'
if 'seed_models' not in globals() or not seed_models:
    seed_models = {seed: PPO.load(str(POOL_DIR_ROBUST / f'agent_beta{BETA}_seed{seed}.zip')) for seed in SEEDS}
    print('시드 모델 로드 완료:', list(seed_models.keys()))


## 9. 평가 — 각 시드를 valid/test에서 실행 (배포 중인 seed=42 포함)

In [ ]:
print('=== 배포 구성(β=-30, α=0.01) 다중 시드 평가 — valid vs test ===\n')

rows_valid, rows_test = {}, {}
curves_valid, curves_test = {}, {}

ew_valid_values, ew_valid_turnovers = equal_weight_backtest(CryptoPortfolioEnv(valid_buckets, **env_kwargs))
ew_test_values,  ew_test_turnovers  = equal_weight_backtest(CryptoPortfolioEnv(test_buckets,  **env_kwargs))
rows_valid['등가중 벤치마크'] = perf_row(ew_valid_values, ew_valid_turnovers)
rows_test['등가중 벤치마크']  = perf_row(ew_test_values,  ew_test_turnovers)
curves_valid['등가중 벤치마크'] = ew_valid_values
curves_test['등가중 벤치마크']  = ew_test_values

print('배포 중인 seed=42 에이전트 평가 중...')
deployed_model = PPO.load(str(DEPLOYED_AGENT_PATH))
v_vals, v_turns = run_beta_smoothed(deployed_model, ALPHA, valid_buckets)
t_vals, t_turns = run_beta_smoothed(deployed_model, ALPHA, test_buckets)
rows_valid['seed=42 (배포 중)'] = perf_row(v_vals, v_turns)
rows_test['seed=42 (배포 중)']  = perf_row(t_vals, t_turns)
curves_valid['seed=42 (배포 중)'] = v_vals
curves_test['seed=42 (배포 중)']  = t_vals

for seed in SEEDS:
    print(f'seed={seed} 평가 중...')
    v_vals, v_turns = run_beta_smoothed(seed_models[seed], ALPHA, valid_buckets)
    t_vals, t_turns = run_beta_smoothed(seed_models[seed], ALPHA, test_buckets)
    rows_valid[f'seed={seed}'] = perf_row(v_vals, v_turns)
    rows_test[f'seed={seed}']  = perf_row(t_vals, t_turns)
    curves_valid[f'seed={seed}'] = v_vals
    curves_test[f'seed={seed}']  = t_vals

valid_df = pd.DataFrame(rows_valid).T
test_df  = pd.DataFrame(rows_test).T

print('\n--- valid(2025 Q4) ---')
print(valid_df.to_string(float_format=lambda x: f'{x:.4f}'))
print('\n--- test(2026~) ---')
print(test_df.to_string(float_format=lambda x: f'{x:.4f}'))

valid_df.to_csv(CRYPTO_DIR / 'crypto_beta_robustness_valid.csv', encoding='utf-8-sig')
test_df.to_csv(CRYPTO_DIR / 'crypto_beta_robustness_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_beta_robustness_valid.csv, crypto_beta_robustness_test.csv')


## 10. 통계 요약 — 신규 5개 시드의 평균·표준편차·범위

In [ ]:
seed_labels = [f'seed={s}' for s in SEEDS]

def agg_stats(df, labels):
    sub = df.loc[labels, ['총 수익률', 'Sharpe', 'MDD']].astype(float)
    return pd.DataFrame({'평균': sub.mean(), '표준편차': sub.std(), '최소': sub.min(), '최대': sub.max()})

print('=== 신규 5개 시드 통계 (valid) ===')
print(agg_stats(valid_df, seed_labels).to_string(float_format=lambda x: f'{x:.4f}'))
print('\n=== 신규 5개 시드 통계 (test) ===')
print(agg_stats(test_df, seed_labels).to_string(float_format=lambda x: f'{x:.4f}'))

print('\n(참고) 배포 중인 seed=42가 이 분포 안에서 어디에 위치하는지 위 두 표의 "seed=42 (배포 중)" 행과 비교할 것.')

agg_stats(valid_df, seed_labels).to_csv(CRYPTO_DIR / 'crypto_beta_robustness_valid_agg.csv', encoding='utf-8-sig')
agg_stats(test_df, seed_labels).to_csv(CRYPTO_DIR / 'crypto_beta_robustness_test_agg.csv', encoding='utf-8-sig')
print('저장 완료: crypto_beta_robustness_valid_agg.csv, crypto_beta_robustness_test_agg.csv')


## 11. 시각화 — 시드별 NAV 곡선

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for label, curve in curves_valid.items():
    lw, ls = (1.8, '--') if '등가중' in label else (1.4, '-')
    axes[0].plot(curve, label=label, lw=lw, ls=ls)
axes[0].set_title('valid(2025 Q4) — 시드별 NAV (초기=1.0)')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

for label, curve in curves_test.items():
    lw, ls = (1.8, '--') if '등가중' in label else (1.4, '-')
    axes[1].plot(curve, label=label, lw=lw, ls=ls)
axes[1].set_title('test(2026~) — 시드별 NAV (초기=1.0)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_beta_robustness_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. 판정 기준

## 14. §12 결과 대응 — 시드 앙상블로 분산을 줄일 수 있는가?

In [ ]:
def run_ensemble_smoothed(models_dict, alpha, buckets, top_k=TOP_K):
    """models_dict의 모든 모델이 매 스텝 독립적으로 낸 목표 비중(top-K+게이트 적용 후)을 평균해
    앙상블 비중을 만들고, 그 위에 §9/§13-F와 동일한 관성 스무딩을 적용한다.
    개별 모델의 top-K 선택이 서로 다를 수 있어 앙상블 자체는 top-K보다 넓은 종목에 걸쳐질 수 있다
    (선택의 다양성을 유지한 채 비중만 평균하는 방식)."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        target_ws, rhos = [], []
        for model in models_dict.values():
            a, _ = model.predict(obs, deterministic=True)
            tw, rho, _ = env.action_to_weights(a)
            target_ws.append(tw)
            rhos.append(rho)
        ensemble_target = np.mean(target_ws, axis=0)
        w = (1 - alpha) * env.weights + alpha * ensemble_target
        selected = np.nonzero(ensemble_target)[0].tolist()
        obs, _, done, _, info = env.step_with_weights(
            w, extra_info={'rho': float(np.mean(rhos)), 'selected': selected}
        )
    return np.array(env.value_history), list(env.turnover_history)


ALL_MODELS = {42: deployed_model, **seed_models}          # 6개 전부
MODELS_NO11 = {k: v for k, v in ALL_MODELS.items() if k != 11}   # 최극단치(seed=11) 제외 5개

print('구성 확인 — 전체 앙상블:', list(ALL_MODELS.keys()), '| seed=11 제외:', list(MODELS_NO11.keys()))


In [ ]:
%%time
print('=== 앙상블 백테스트 — valid vs test ===\n')

ens_rows_valid, ens_rows_test = {}, {}
ens_curves_valid, ens_curves_test = {}, {}

print('전체 6개 앙상블 실행 중 (valid)...')
v_vals, v_turns = run_ensemble_smoothed(ALL_MODELS, ALPHA, valid_buckets)
print('전체 6개 앙상블 실행 중 (test)...')
t_vals, t_turns = run_ensemble_smoothed(ALL_MODELS, ALPHA, test_buckets)
ens_rows_valid['앙상블(6개 전부)'] = perf_row(v_vals, v_turns)
ens_rows_test['앙상블(6개 전부)']  = perf_row(t_vals, t_turns)
ens_curves_valid['앙상블(6개 전부)'] = v_vals
ens_curves_test['앙상블(6개 전부)']  = t_vals

print('seed=11 제외 5개 앙상블 실행 중 (valid)...')
v_vals2, v_turns2 = run_ensemble_smoothed(MODELS_NO11, ALPHA, valid_buckets)
print('seed=11 제외 5개 앙상블 실행 중 (test)...')
t_vals2, t_turns2 = run_ensemble_smoothed(MODELS_NO11, ALPHA, test_buckets)
ens_rows_valid['앙상블(seed=11 제외 5개)'] = perf_row(v_vals2, v_turns2)
ens_rows_test['앙상블(seed=11 제외 5개)']  = perf_row(t_vals2, t_turns2)
ens_curves_valid['앙상블(seed=11 제외 5개)'] = v_vals2
ens_curves_test['앙상블(seed=11 제외 5개)']  = t_vals2

# 참고용 — 등가중 벤치마크와 개별 seed=42(배포 중)도 같은 표에 나란히
ens_rows_valid['등가중 벤치마크']  = rows_valid['등가중 벤치마크']
ens_rows_test['등가중 벤치마크']   = rows_test['등가중 벤치마크']
ens_rows_valid['seed=42 단독(현재 배포)'] = rows_valid['seed=42 (배포 중)']
ens_rows_test['seed=42 단독(현재 배포)']  = rows_test['seed=42 (배포 중)']

ens_valid_df = pd.DataFrame(ens_rows_valid).T
ens_test_df  = pd.DataFrame(ens_rows_test).T

print('\n--- valid(2025 Q4) ---')
print(ens_valid_df.to_string(float_format=lambda x: f'{x:.4f}'))
print('\n--- test(2026~) ---')
print(ens_test_df.to_string(float_format=lambda x: f'{x:.4f}'))

ens_valid_df.to_csv(CRYPTO_DIR / 'crypto_beta_ensemble_valid.csv', encoding='utf-8-sig')
ens_test_df.to_csv(CRYPTO_DIR / 'crypto_beta_ensemble_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_beta_ensemble_valid.csv, crypto_beta_ensemble_test.csv')


In [ ]:
# 낙차(consistency) 비교 — Sharpe가 valid→test 사이에 얼마나 흔들렸는지 개별 시드 vs 앙상블
print('=== valid→test Sharpe 낙차 비교 (절대값 작을수록 안정적) ===\n')

gap_rows = {}
for label in seed_labels + ['seed=42 (배포 중)']:
    v_sh = valid_df.loc[label, 'Sharpe']
    t_sh = test_df.loc[label, 'Sharpe']
    gap_rows[label] = {'valid Sharpe': v_sh, 'test Sharpe': t_sh, '낙차(|test-valid|)': abs(t_sh - v_sh)}

for label in ['앙상블(6개 전부)', '앙상블(seed=11 제외 5개)']:
    v_sh = ens_valid_df.loc[label, 'Sharpe']
    t_sh = ens_test_df.loc[label, 'Sharpe']
    gap_rows[label] = {'valid Sharpe': v_sh, 'test Sharpe': t_sh, '낙차(|test-valid|)': abs(t_sh - v_sh)}

gap_df = pd.DataFrame(gap_rows).T.sort_values('낙차(|test-valid|)')
print(gap_df.to_string(float_format=lambda x: f'{x:.4f}'))
gap_df.to_csv(CRYPTO_DIR / 'crypto_beta_ensemble_gap_comparison.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_beta_ensemble_gap_comparison.csv')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for label, curve in ens_curves_valid.items():
    axes[0].plot(curve, label=label, lw=2.0)
axes[0].plot(curves_valid['등가중 벤치마크'], label='등가중 벤치마크', lw=1.5, ls='--', color='black')
axes[0].set_title('valid(2025 Q4) — 앙상블 vs 등가중')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

for label, curve in ens_curves_test.items():
    axes[1].plot(curve, label=label, lw=2.0)
axes[1].plot(curves_test['등가중 벤치마크'], label='등가중 벤치마크', lw=1.5, ls='--', color='black')
axes[1].set_title('test(2026~) — 앙상블 vs 등가중')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_beta_ensemble_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 15. 판정 기준

## 16. Deflated Sharpe Ratio (DSR) — walk-forward 대신 다중시드 결과에 통계적 엄밀성 부여

In [ ]:
from scipy import stats

def deflated_sharpe_ratio(sr_hat_per_step, sr_trials_per_step, n_obs, returns_for_moments):
    """Bailey & Lopez de Prado (2014) Deflated Sharpe Ratio.

    sr_hat_per_step: 검증 대상 시드의 스텝당(연환산 전) Sharpe.
    sr_trials_per_step: 같은 실험에서 시도한 N개 시드의 스텝당 Sharpe 배열(다중검정 보정용).
    n_obs: 수익률 관측치 개수(원 스텝 수, 연환산 배수 아님).
    returns_for_moments: 검증 대상 시드의 스텝별 수익률(왜도·첨도 계산용).
    반환: dict(SR*, DSR, N, sr_std)
    """
    N = len(sr_trials_per_step)
    sr_std = np.std(sr_trials_per_step, ddof=1) if N > 1 else 0.0
    euler_mascheroni = 0.5772156649

    if N > 1 and sr_std > 0:
        sr_star = sr_std * (
            (1 - euler_mascheroni) * stats.norm.ppf(1 - 1.0 / N)
            + euler_mascheroni * stats.norm.ppf(1 - 1.0 / (N * np.e))
        )
    else:
        sr_star = 0.0

    skew = stats.skew(returns_for_moments)
    kurt = stats.kurtosis(returns_for_moments, fisher=False)  # 정규분포=3(비-초과 첨도)

    denom = np.sqrt(max(1 - skew * sr_hat_per_step + ((kurt - 1) / 4) * sr_hat_per_step ** 2, 1e-12))
    z = (sr_hat_per_step - sr_star) * np.sqrt(max(n_obs - 1, 1)) / denom
    dsr = float(stats.norm.cdf(z))
    return {'SR*(기대 최댓값)': sr_star, 'DSR': dsr, 'N(시행 수)': N, 'σ_SR': sr_std,
            '왜도': skew, '첨도': kurt, '관측치 수': n_obs}


def per_step_sharpe_and_returns(nav_curve, steps_per_year=48 * 365):
    rets = np.diff(nav_curve) / nav_curve[:-1]
    sr_per_step = rets.mean() / (rets.std() + 1e-8)   # 연환산 안 한 스텝당 Sharpe (DSR 공식은 이 스케일을 씀)
    return sr_per_step, rets


print('DSR 함수 정의 완료')


In [ ]:
print('=== [test 구간] 6개 시드 DSR 계산 — seed=42(배포)가 다중검정 보정 후에도 유의한가 ===\n')

seed_order = [42] + SEEDS   # curves_test의 라벨과 매칭되는 순서
label_map = {42: 'seed=42 (배포 중)', **{s: f'seed={s}' for s in SEEDS}}

sr_per_step_all, rets_all = {}, {}
for s in seed_order:
    label = label_map[s]
    sr, rets = per_step_sharpe_and_returns(curves_test[label])
    sr_per_step_all[s] = sr
    rets_all[s] = rets

sr_trials = np.array([sr_per_step_all[s] for s in seed_order])
n_obs = len(rets_all[42])

dsr_result_deployed = deflated_sharpe_ratio(
    sr_hat_per_step=sr_per_step_all[42],
    sr_trials_per_step=sr_trials,
    n_obs=n_obs,
    returns_for_moments=rets_all[42],
)
print('[seed=42, 배포 중] DSR 결과:')
for k, v in dsr_result_deployed.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# 참고: seed=11(최댓값 Sharpe 2.567을 기록했던 시드)도 같은 방식으로 계산 — "제일 좋아 보였던 시드"의 DSR
dsr_result_seed11 = deflated_sharpe_ratio(
    sr_hat_per_step=sr_per_step_all[11],
    sr_trials_per_step=sr_trials,
    n_obs=n_obs,
    returns_for_moments=rets_all[11],
)
print('\n[seed=11, raw Sharpe 최댓값] DSR 결과 (참고용 — 이 시드를 배포했다면 어땠을지):')
for k, v in dsr_result_seed11.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

dsr_summary = pd.DataFrame({
    'seed=42(배포)': dsr_result_deployed,
    'seed=11(raw 최댓값)': dsr_result_seed11,
}).T
dsr_summary.to_csv(CRYPTO_DIR / 'crypto_beta_dsr.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_beta_dsr.csv')


### 해석 기준 (2026-07-29 실행 결과 반영, 사전 예측 정정)

## 17. N 확대 — 시드 6개 → 16개로 늘려 통계적 검정력 보강 (`paper/GAP_ANALYSIS.md` §1.4)

In [ ]:
SEEDS_EXTRA = [66, 77, 88, 99, 110, 121, 132, 143, 154, 165]  # 기존 SEEDS=[11,22,33,44,55]와 겹치지 않는 신규 10개
POOL_DIR_ROBUST.mkdir(parents=True, exist_ok=True)
print('신규 시드:', SEEDS_EXTRA)


In [ ]:
# 재실행용 로더 (idempotent)
if 'seed_models_extra' not in globals() or not seed_models_extra:
    seed_models_extra = {seed: PPO.load(str(POOL_DIR_ROBUST / f'agent_beta{BETA}_seed{seed}.zip')) for seed in SEEDS_EXTRA}
    print('신규 시드 모델 로드 완료:', list(seed_models_extra.keys()))


### 17-B. 신규 10개 시드 평가 (valid/test) + 기존 6개와 통합

In [ ]:
print('=== 신규 10개 시드 평가 — valid vs test ===\n')

for seed in SEEDS_EXTRA:
    print(f'seed={seed} 평가 중...')
    v_vals, v_turns = run_beta_smoothed(seed_models_extra[seed], ALPHA, valid_buckets)
    t_vals, t_turns = run_beta_smoothed(seed_models_extra[seed], ALPHA, test_buckets)
    rows_valid[f'seed={seed}'] = perf_row(v_vals, v_turns)
    rows_test[f'seed={seed}']  = perf_row(t_vals, t_turns)
    curves_valid[f'seed={seed}'] = v_vals
    curves_test[f'seed={seed}']  = t_vals

ALL_SEEDS_16 = [42] + SEEDS + SEEDS_EXTRA   # 배포(42) + 기존 5개 + 신규 10개 = 16
all_labels_16 = [label_map.get(s, f'seed={s}') if 'label_map' in globals() else
                 ('seed=42 (배포 중)' if s == 42 else f'seed={s}') for s in ALL_SEEDS_16]

valid_df_16 = pd.DataFrame({lbl: rows_valid[lbl] for lbl in all_labels_16}).T
test_df_16  = pd.DataFrame({lbl: rows_test[lbl]  for lbl in all_labels_16}).T

print('\n--- valid(2025 Q4), N=16 전부 ---')
print(valid_df_16.to_string(float_format=lambda x: f'{x:.4f}'))
print('\n--- test(2026~), N=16 전부 ---')
print(test_df_16.to_string(float_format=lambda x: f'{x:.4f}'))

valid_df_16.to_csv(CRYPTO_DIR / 'crypto_beta_robustness_valid_N16.csv', encoding='utf-8-sig')
test_df_16.to_csv(CRYPTO_DIR / 'crypto_beta_robustness_test_N16.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_beta_robustness_valid_N16.csv, crypto_beta_robustness_test_N16.csv')


### 17-C. N=16 기준 통계 요약 + DSR 재계산

In [ ]:
non_deployed_labels_16 = [lbl for lbl in all_labels_16 if lbl != 'seed=42 (배포 중)']

def agg_stats_16(df, labels):
    sub = df.loc[labels, ['총 수익률', 'Sharpe', 'MDD']].astype(float)
    return pd.DataFrame({'평균': sub.mean(), '표준편차': sub.std(), '최소': sub.min(), '최대': sub.max()})

print('=== N=15(배포 제외) 통계 (valid) ===')
print(agg_stats_16(valid_df_16, non_deployed_labels_16).to_string(float_format=lambda x: f'{x:.4f}'))
print('\n=== N=15(배포 제외) 통계 (test) ===')
print(agg_stats_16(test_df_16, non_deployed_labels_16).to_string(float_format=lambda x: f'{x:.4f}'))

# --- DSR 재계산 (N=16) ---
sr_per_step_all_16 = {}
rets_all_16 = {}
for s, lbl in zip(ALL_SEEDS_16, all_labels_16):
    sr, rets = per_step_sharpe_and_returns(curves_test[lbl])
    sr_per_step_all_16[s] = sr
    rets_all_16[s] = rets

sr_trials_16 = np.array([sr_per_step_all_16[s] for s in ALL_SEEDS_16])
n_obs_16 = len(rets_all_16[42])

dsr_deployed_16 = deflated_sharpe_ratio(
    sr_hat_per_step=sr_per_step_all_16[42],
    sr_trials_per_step=sr_trials_16,
    n_obs=n_obs_16,
    returns_for_moments=rets_all_16[42],
)
print('\n=== [N=16] seed=42(배포) DSR 재계산 ===')
for k, v in dsr_deployed_16.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

best_seed_16 = max(ALL_SEEDS_16, key=lambda s: sr_per_step_all_16[s])
dsr_best_16 = deflated_sharpe_ratio(
    sr_hat_per_step=sr_per_step_all_16[best_seed_16],
    sr_trials_per_step=sr_trials_16,
    n_obs=n_obs_16,
    returns_for_moments=rets_all_16[best_seed_16],
)
print(f'\n=== [N=16] raw Sharpe 최댓값 시드(seed={best_seed_16}) DSR 재계산 ===')
for k, v in dsr_best_16.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

pd.DataFrame({
    f'seed=42(배포, N=16)': dsr_deployed_16,
    f'seed={best_seed_16}(raw 최댓값, N=16)': dsr_best_16,
}).T.to_csv(CRYPTO_DIR / 'crypto_beta_dsr_N16.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_beta_dsr_N16.csv')


### 판정 기준